# Head-to-Head Report

A supporter-friendly historical comparison of two teams in one Rugby Tracker competition. Set the inputs below and run all cells. Names are matched case-insensitively. Use `SEASON = "All Seasons"` to include every stored season; when only one season exists it is selected automatically.

In [ ]:
COMPETITION = "PWR"
SEASON = "All Seasons"
TEAM_A = "Harlequins Women"
TEAM_B = "Sale Sharks Women"

## Setup and data loading

In [ ]:
from __future__ import annotations

import html
import os
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_rows", 200)
plt.style.use("seaborn-v0_8-whitegrid")
TEAM_A_COLOUR, TEAM_B_COLOUR, DRAW_COLOUR = "#1f77b4", "#d62728", "#888888"

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "rugby_tracker").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from within the RugbyTracker repository.")

def cards(items):
    boxes = "".join(
        f"<div style='padding:12px 16px;border:1px solid #ddd;border-radius:8px;min-width:130px'>"
        f"<small>{html.escape(str(label))}</small><br><b style='font-size:1.2em'>{html.escape(str(value))}</b></div>"
        for label, value in items
    )
    display(HTML(f"<div style='display:flex;gap:10px;flex-wrap:wrap;margin:8px 0 18px'>{boxes}</div>"))

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DB_PATH = Path(os.environ.get("RUGBY_TRACKER_DB", REPO_ROOT / "data" / "rugbytracker.db")).expanduser()
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Rugby Tracker database not found: {DB_PATH}")
connection = sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
print(f"Using database: {DB_PATH}")

In [ ]:
competition_rows = [dict(row) for row in connection.execute(
    "SELECT id, name, season FROM competitions WHERE name = ? COLLATE NOCASE ORDER BY season",
    (COMPETITION.strip(),),
)]
if not competition_rows:
    available = pd.read_sql_query("SELECT name, season FROM competitions ORDER BY name, season", connection)
    raise ValueError(f"Competition not found: {COMPETITION!r}. Available:\n{available.to_string(index=False)}")
competition_name = competition_rows[0]["name"]
seasons = [str(row["season"]) for row in competition_rows]
requested_season = None if SEASON is None else str(SEASON).strip()
all_seasons = requested_season is not None and requested_season.casefold() == "all seasons".casefold()
if len(competition_rows) == 1:
    selected_rows = competition_rows
    selected_season = seasons[0]
elif all_seasons:
    selected_rows = competition_rows
    selected_season = "All Seasons"
elif requested_season:
    selected_rows = [row for row in competition_rows if str(row["season"]).casefold() == requested_season.casefold()]
    if not selected_rows:
        raise ValueError(f"Season {SEASON!r} not found for {competition_name}. Available: {', '.join(seasons)}")
    selected_season = str(selected_rows[0]["season"])
else:
    raise ValueError(f"{competition_name} has multiple seasons ({', '.join(seasons)}); set SEASON or use 'All Seasons'.")

competition_ids = [row["id"] for row in selected_rows]
placeholders = ",".join("?" for _ in competition_ids)
matches_sql = f"""
SELECT m.id, m.round, m.match_date, m.kickoff_time, m.home_score, m.away_score,
       m.home_tries, m.away_tries, c.name AS competition, c.season,
       h.name AS home_team, a.name AS away_team, COALESCE(v.name, '—') AS venue
FROM matches m
JOIN competitions c ON c.id = m.competition_id
JOIN teams h ON h.id = m.home_team_id
JOIN teams a ON a.id = m.away_team_id
LEFT JOIN venues v ON v.id = m.venue_id
WHERE m.competition_id IN ({placeholders})
ORDER BY m.match_date, COALESCE(m.kickoff_time, ''), m.id
"""
all_matches = [dict(row) for row in connection.execute(matches_sql, competition_ids)]
team_names = sorted({m["home_team"] for m in all_matches} | {m["away_team"] for m in all_matches}, key=str.casefold)

def resolve_team(value):
    matches = [name for name in team_names if name.casefold() == value.strip().casefold()]
    if not matches:
        raise ValueError(f"Team not found: {value!r}. Available for {competition_name}: {', '.join(team_names)}")
    return matches[0]

team_a, team_b = resolve_team(TEAM_A), resolve_team(TEAM_B)
if team_a == team_b:
    raise ValueError("TEAM_A and TEAM_B must be different teams.")
meetings = [m for m in all_matches if {m["home_team"], m["away_team"]} == {team_a, team_b} and m["home_score"] is not None and m["away_score"] is not None]
if not meetings:
    raise ValueError(f"No completed meetings found between {team_a} and {team_b} in {competition_name} ({selected_season}).")
print(f"Available seasons: {', '.join(seasons)}")
print(f"Available teams: {', '.join(team_names)}")

In [ ]:
def meeting_view(match):
    a_is_home = match["home_team"] == team_a
    a_points = int(match["home_score"] if a_is_home else match["away_score"])
    b_points = int(match["away_score"] if a_is_home else match["home_score"])
    a_tries = match["home_tries"] if a_is_home else match["away_tries"]
    b_tries = match["away_tries"] if a_is_home else match["home_tries"]
    winner = team_a if a_points > b_points else team_b if b_points > a_points else "Draw"
    return {
        "Date": pd.to_datetime(match["match_date"]), "Competition": match["competition"],
        "Season": str(match["season"]), "Round": match["round"] or "—", "Venue": match["venue"],
        "Home Team": match["home_team"], "Away Team": match["away_team"],
        "Score": f"{match['home_score']}–{match['away_score']}", "Winner": winner,
        f"{team_a} Points": a_points, f"{team_b} Points": b_points,
        f"{team_a} Tries": int(a_tries) if a_tries is not None else pd.NA,
        f"{team_b} Tries": int(b_tries) if b_tries is not None else pd.NA,
        "Margin": a_points - b_points, "Combined Points": a_points + b_points,
        "Team A Hosted": a_is_home,
    }

results = pd.DataFrame(meeting_view(match) for match in meetings).sort_values("Date", kind="stable").reset_index(drop=True)
results["Meeting"] = range(1, len(results) + 1)
a_points_col, b_points_col = f"{team_a} Points", f"{team_b} Points"
a_tries_col, b_tries_col = f"{team_a} Tries", f"{team_b} Tries"

def team_record(team, opponent, points_col, opponent_points_col, tries_col, opponent_tries_col):
    return {
        "Wins": int((results["Winner"] == team).sum()),
        "Draws": int((results["Winner"] == "Draw").sum()),
        "Losses": int((results["Winner"] == opponent).sum()),
        "Win Percentage": f"{(results['Winner'] == team).mean():.1%}",
        "Points Scored": int(results[points_col].sum()),
        "Points Conceded": int(results[opponent_points_col].sum()),
        "Tries Scored": int(results[tries_col].sum()) if results[tries_col].notna().all() else "—",
        "Tries Conceded": int(results[opponent_tries_col].sum()) if results[opponent_tries_col].notna().all() else "—",
    }

record_a = team_record(team_a, team_b, a_points_col, b_points_col, a_tries_col, b_tries_col)
record_b = team_record(team_b, team_a, b_points_col, a_points_col, b_tries_col, a_tries_col)

## Fixture summary

In [ ]:
display(HTML(f"<h2>{html.escape(team_a)} <span style='color:#888'>v</span> {html.escape(team_b)}</h2>"))
cards([("Competition", competition_name), ("Season", selected_season), ("Team A", team_a), ("Team B", team_b), ("Meetings", len(results))])

## Overall head-to-head record

In [ ]:
overall = pd.DataFrame({team_a: record_a, team_b: record_b})
display(overall.style.set_caption(f"{team_a} v {team_b}"))

## Home and away record

In [ ]:
def hosted_record(host, subset, host_points, visitor_points):
    visitor = team_b if host == team_a else team_a
    return {
        "Host": host, "Meetings": len(subset),
        "Host Wins": int((subset["Winner"] == host).sum()),
        "Draws": int((subset["Winner"] == "Draw").sum()),
        "Host Losses": int((subset["Winner"] == visitor).sum()),
        "Avg Points Scored": round(subset[host_points].mean(), 1) if len(subset) else "—",
        "Avg Points Conceded": round(subset[visitor_points].mean(), 1) if len(subset) else "—",
    }

a_hosted = results[results["Team A Hosted"]]
b_hosted = results[~results["Team A Hosted"]]
home_away = pd.DataFrame([hosted_record(team_a, a_hosted, a_points_col, b_points_col), hosted_record(team_b, b_hosted, b_points_col, a_points_col)])
display(home_away.style.hide(axis="index"))

## Match history

In [ ]:
history_columns = ["Date", "Competition", "Season", "Round", "Venue", "Home Team", "Away Team", "Score", "Winner"]
def highlight_winner(value):
    colour = TEAM_A_COLOUR if value == team_a else TEAM_B_COLOUR if value == team_b else DRAW_COLOUR
    return f"color:{colour};font-weight:bold"
display(results[history_columns].style.hide(axis="index").format({"Date": "{:%d %b %Y}"}).map(highlight_winner, subset=["Winner"]))

## Results timeline and scoring trends

In [ ]:
labels = results["Date"].dt.strftime("%d %b %Y")
x = results["Meeting"]
outcome_colours = [TEAM_A_COLOUR if winner == team_a else TEAM_B_COLOUR if winner == team_b else DRAW_COLOUR for winner in results["Winner"]]
fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
axes[0].scatter(x, [0] * len(results), c=outcome_colours, s=120, zorder=3)
axes[0].axhline(0, color="#bbbbbb", linewidth=1)
axes[0].set(title="Results timeline", yticks=[])
for meeting, winner in zip(x, results["Winner"]):
    axes[0].annotate(winner if winner == "Draw" else f"{winner} win", (meeting, 0), xytext=(0, 10), textcoords="offset points", ha="center", fontsize=8)
axes[1].plot(x, results[a_points_col], marker="o", linewidth=2, color=TEAM_A_COLOUR, label=team_a)
axes[1].plot(x, results[b_points_col], marker="o", linewidth=2, color=TEAM_B_COLOUR, label=team_b)
axes[1].set(title="Points scored by meeting", ylabel="Points"); axes[1].legend()
axes[2].bar(x, results["Margin"], color=outcome_colours)
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set(title=f"Winning margin (positive = {team_a}, negative = {team_b})", xlabel="Meeting date", ylabel="Margin", xticks=x, xticklabels=labels)
axes[2].tick_params(axis="x", rotation=35)
plt.tight_layout(); plt.show()

## Average scores and notable matches

In [ ]:
cards([(f"Average {team_a} score", f"{results[a_points_col].mean():.1f}"), (f"Average {team_b} score", f"{results[b_points_col].mean():.1f}"), ("Average combined score", f"{results['Combined Points'].mean():.1f}")])

def describe(row):
    return f"{row['Date']:%d %b %Y} · {row['Competition']} ({row['Season']}) · {row['Venue']} · {row['Home Team']} {row['Score']} {row['Away Team']}"

a_wins = results[results["Winner"] == team_a]
b_wins = results[results["Winner"] == team_b]
largest_a = a_wins.loc[a_wins["Margin"].idxmax()] if not a_wins.empty else None
largest_b = b_wins.loc[b_wins["Margin"].idxmin()] if not b_wins.empty else None
highest = results.loc[results["Combined Points"].idxmax()]
highlights = pd.DataFrame([
    (f"Largest {team_a} victory", describe(largest_a) if largest_a is not None else "No victories"),
    (f"Largest {team_b} victory", describe(largest_b) if largest_b is not None else "No victories"),
    ("Highest-scoring match", describe(highest)),
], columns=["Highlight", "Match"]).set_index("Highlight")
display(highlights)

## Closest matches and current streak

In [ ]:
close_matches = results[results["Margin"].abs() <= 7].copy()
close_matches["Three Points or Fewer"] = close_matches["Margin"].abs() <= 3
if close_matches.empty:
    display(HTML("<p>No meetings were decided by one score (seven points) or fewer.</p>"))
else:
    display(close_matches[["Date", "Home Team", "Away Team", "Score", "Winner", "Margin", "Three Points or Fewer"]].style.hide(axis="index").format({"Date": "{:%d %b %Y}", "Margin": lambda value: abs(int(value))}))

latest_winner = results.iloc[-1]["Winner"]
if latest_winner == "Draw":
    streak_text = "The teams drew their latest meeting."
else:
    streak = 0
    for winner in reversed(results["Winner"].tolist()):
        if winner != latest_winner:
            break
        streak += 1
    meeting_word = "meeting" if streak == 1 else "meetings"
    streak_text = f"{latest_winner} {'won the latest meeting' if streak == 1 else f'has won the last {streak} meetings'}."
display(HTML(f"<h3>Current streak</h3><p style='font-size:1.15em'><b>{html.escape(streak_text)}</b></p>"))

---
**Notes.** Only completed fixtures are included. A one-score match is defined as a margin of seven points or fewer; the closest-match table separately flags margins of three points or fewer. Missing try data is shown as `—` rather than treated as zero. Tied notable results use the first match chronologically. Export with **File → Save and Export Notebook As → PDF**, or run `jupyter nbconvert --to webpdf "Head-to-Head Report.ipynb"` from the `reports` folder.